In [1]:
#!pip install pygbif pyobis pandas geopandas shapely pyproj
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
from shapely.geometry import Point
from pygbif import occurrences as gbif_occ
from pyobis import occurrences as obis_occ


In [3]:
# Set time window
START_YEAR = 2012
END_YEAR = pd.Timestamp.today().year


# Occurrence data importing functions
def get_gbif_occurrences(species, limit=100):
    res = gbif_occ.search(
        scientificName=species,
        hasCoordinate=True,
        year=f"{START_YEAR},{END_YEAR}",
        limit=limit)

    df = pd.DataFrame(res["results"])
    df = df[["decimalLongitude", "decimalLatitude", "eventDate"]]
    df["source"] = "GBIF"

    return df

def get_obis_occurrences(species, page_size=100):
    url = "https://api.obis.org/v3/occurrence"
    all_records = []
    offset = 0
    total = None
    
    while True:
        params = {"scientificname": species,
            "year": f"{START_YEAR},{END_YEAR}",
            "size": page_size, "offset": offset}

        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        if total is None:
            total = data.get("total", 0)

        records = data.get("results", [])

        if not records:
            break

        all_records.extend(records)
        offset += page_size

        if offset >= total:
            break

    df = pd.DataFrame(all_records)
    df = df[["decimalLongitude", "decimalLatitude", "eventDate"]]
    df["source"] = "OBIS"

    return df

# Import blue shark data
species = "Prionace glauca"

gbif_df = get_gbif_occurrences(species)
obis_df = get_obis_occurrences(species)
occurrences = pd.concat([gbif_df, obis_df], ignore_index=True)

In [4]:
# Convert to GeoDataFrame
occurrences = occurrences.dropna(
    subset=["decimalLongitude", "decimalLatitude"])

gdf = gpd.GeoDataFrame(occurrences,
    geometry=gpd.points_from_xy(occurrences.decimalLongitude, occurrences.decimalLatitude),
    crs="EPSG:4326")

# Load IUCN blue shark range map
iucn_range = gpd.read_file("shark_dist.shp")

# Ensure map is in WGS84
# if iucn_range.crs != "EPSG:4326":  iucn_range = iucn_range.to_crs("EPSG:4326")

gdf_clean = gpd.sjoin(gdf, iucn_range, predicate="within", how="inner").drop(columns="index_right")



In [ ]:
# Load Earth data
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

# Plot
fig, ax = plt.subplots(figsize=(14, 8))

# World basemap
world.plot(ax=ax, color="lightgray", edgecolor="white")

# IUCN range
iucn_range.plot(ax=ax, color="lightblue", edgecolor="blue",
    alpha=0.4, label="IUCN range")

# Cleaned occurrences
gdf_clean.plot(ax=ax, markersize=5, color="red",
    alpha=0.6, label="Occurrences")

ax.set_title("Prionace glauca – Cleaned Occurrences", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()

plt.tight_layout()
plt.show()
